# 00 — Native data exploration and statistical intake

**Purpose:** understand what the source actually contains before translating it or
building a detector. This is an evidence notebook, not a generic chart gallery.

It answers five practical questions:

1. What is one observation, which features exist, and what do they mean?
2. Which apparent patterns are entity, peer-group, time, or measurement effects?
3. How much independent time-series information is present after autocorrelation?
4. What data-quality and nuisance modes can masquerade as anomalies?
5. Would even a simple label-free score create a manageable alert volume?

The notebook supports `telecom` and `petrobras_3w`. It uses the native source, writes
small reports and figures to Drive, and never copies the large source panel.


## Statistical discipline

- **Seal before looking:** entities, time ranges, and 3W files are deterministically
  assigned to exploration or holdout before feature analysis.
- **No label-shaped reasoning before the freeze:** truth values and 3W event folders
  are not inspected in the operational sections.
- **Three different absences stay different:** a present row with a null value, an
  expected observation that is absent, and an entity outside its service period.
- **Dependence is respected:** rows in one series are not treated as independent
  replicates. We report effective sample size and entity-balanced associations.
- **No silent imputation:** temporal and recurrence screens use observed pairs only.
- **Exploration is descriptive:** synthetic telecom prevalence and event-conditioned
  3W files do not estimate deployment incident prevalence.


## 1. Setup, paths and reproducible controls


In [ ]:
import configparser
import hashlib
import json
import math
import os
import re
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq
import scipy
from scipy import stats
from statsmodels.stats.stattools import medcouple
from IPython.display import display

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path(os.getenv(
    "ANOMALY_DRIVE_ROOT",
    "/content/drive/MyDrive/anomaly_detection",
))
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DRIVE_ROOT / "research" / "week1",
))
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from week1_core import (
    CORE_VERSION,
    discover_telecom,
    discover_threew,
    native_path,
)

EDA_SECTOR = os.getenv("EDA_SECTOR", "telecom")
assert EDA_SECTOR in {"telecom", "petrobras_3w"}

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
EDA_RUN_ID = os.getenv("EDA_RUN_ID", f"{EDA_SECTOR}_native_eda_v2_{RUN_STAMP}")
RANDOM_SEED = int(os.getenv("EDA_RANDOM_SEED", "42"))
DEV_ENTITY_FRACTION = float(os.getenv("EDA_DEV_ENTITY_FRACTION", "0.70"))
DEV_TIME_FRACTION = float(os.getenv("EDA_DEV_TIME_FRACTION", "0.60"))
THREEW_DEV_FILE_FRACTION = float(
    os.getenv("EDA_THREEW_DEV_FILE_FRACTION", "0.60")
)
SAMPLE_ROWS = int(os.getenv("EDA_SAMPLE_ROWS", "200000"))
THREEW_FILE_COUNT = int(os.getenv("EDA_THREEW_FILE_COUNT", "20"))
THREEW_ROWS_PER_FILE = int(
    os.getenv("EDA_THREEW_ROWS_PER_FILE", "10000")
)
LONGITUDINAL_ENTITY_COUNT = int(
    os.getenv("EDA_LONGITUDINAL_ENTITY_COUNT", "12")
)
LONGITUDINAL_FILE_COUNT = int(
    os.getenv("EDA_LONGITUDINAL_FILE_COUNT", "6")
)
SERIES_PLOT_COUNT = int(os.getenv("EDA_SERIES_PLOT_COUNT", "2"))
SERIES_MAX_POINTS = int(os.getenv("EDA_SERIES_MAX_POINTS", "3000"))
BOOTSTRAP_REPS = int(os.getenv("EDA_BOOTSTRAP_REPS", "50"))
INCLUDE_TRUTH_AUDIT = os.getenv("EDA_INCLUDE_TRUTH", "1") == "1"

TELECOM_SOURCE = Path(os.getenv(
    "TELECOM_SOURCE_ROOT",
    str(
        DRIVE_ROOT / "Full dataset"
        if (DRIVE_ROOT / "Full dataset").exists()
        else DRIVE_ROOT
    ),
))
THREEW_SOURCE = Path(os.getenv(
    "THREEW_SOURCE_ROOT",
    str(
        DRIVE_ROOT / "sources" / "petrobras_3w" / "2.0.0"
        / "raw" / "3w_dataset_2.0.0"
    ),
))
SOURCE_ROOT = TELECOM_SOURCE if EDA_SECTOR == "telecom" else THREEW_SOURCE
OUTPUT = DRIVE_ROOT / "outputs" / "exploration" / EDA_SECTOR / EDA_RUN_ID
PRELABEL = OUTPUT / "prelabel"
FIGURES = PRELABEL / "figures"
LABEL_AUDIT = OUTPUT / "label_audit"
if OUTPUT.exists():
    raise FileExistsError(
        f"Run directory already exists: {OUTPUT}\n"
        "Choose a new EDA_RUN_ID; frozen evidence is never overwritten."
    )
FIGURES.mkdir(parents=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 180)
rng = np.random.default_rng(RANDOM_SEED)

display(pd.Series({
    "sector": EDA_SECTOR,
    "source": str(SOURCE_ROOT),
    "output": str(OUTPUT),
    "development_entity_fraction": DEV_ENTITY_FRACTION,
    "development_time_fraction": DEV_TIME_FRACTION,
    "operational_sample_rows": SAMPLE_ROWS,
    "truth_audit_after_freeze": INCLUDE_TRUTH_AUDIT,
    "random_seed": RANDOM_SEED,
}, name="value").to_frame())


In [ ]:
def hash_fraction(value, namespace):
    raw = f"{RANDOM_SEED}|{namespace}|{value}".encode("utf-8")
    return int(hashlib.sha256(raw).hexdigest()[:16], 16) / 16**16


def opaque_id(value, prefix="id"):
    raw = f"{RANDOM_SEED}|{value}".encode("utf-8")
    return f"{prefix}_{hashlib.sha256(raw).hexdigest()[:16]}"


def parquet_shape(path):
    parquet = pq.ParquetFile(path)
    return parquet.metadata.num_rows, len(parquet.schema_arrow.names)


def csv_shape(path):
    columns = len(pd.read_csv(path, nrows=0).columns)
    with Path(path).open("r", encoding="utf-8", errors="replace") as handle:
        rows = max(0, sum(1 for _ in handle) - 1)
    return rows, columns


def timestamp_bounds(parquet, field):
    if field not in parquet.schema.names:
        return pd.NaT, pd.NaT
    column_index = parquet.schema.names.index(field)
    starts, ends = [], []
    for index in range(parquet.num_row_groups):
        statistics = parquet.metadata.row_group(index).column(
            column_index
        ).statistics
        if statistics is not None and statistics.has_min_max:
            starts.append(pd.to_datetime(statistics.min, utc=True))
            ends.append(pd.to_datetime(statistics.max, utc=True))
    return (
        min(starts) if starts else pd.NaT,
        max(ends) if ends else pd.NaT,
    )


def threew_source_kind(path):
    if path.stem.startswith("WELL-"):
        return "real"
    if path.stem.startswith("SIMULATED_"):
        return "simulated"
    if path.stem.startswith("DRAWN_"):
        return "hand_drawn"
    return "other"


def read_threew(path, columns):
    frame = pd.read_parquet(path, columns=columns)
    if "timestamp" not in frame.columns:
        frame = frame.reset_index()
    return frame


def write_csv(frame, name, folder=PRELABEL, index=False):
    path = folder / name
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=index)
    return path


def write_json(payload, name, folder=PRELABEL):
    path = folder / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(payload, indent=2, sort_keys=True, default=str) + "\n",
        encoding="utf-8",
    )
    return path


def save_figure(figure, filename):
    figure.tight_layout()
    figure.savefig(FIGURES / filename, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(figure)


def allocate_integer(total, weights):
    weights = np.asarray(weights, dtype=float)
    if weights.sum() == 0 or total <= 0:
        return np.zeros(len(weights), dtype=int)
    total = min(int(total), int(weights.sum()))
    raw = total * weights / weights.sum()
    allocation = np.floor(raw).astype(int)
    capacity = weights.astype(int) - allocation
    order = np.argsort(-(raw - allocation))
    remaining = total - allocation.sum()
    for index in order:
        if remaining == 0:
            break
        if capacity[index] > 0:
            allocation[index] += 1
            remaining -= 1
    return allocation


def sha256_tree(root):
    hashes = {}
    for path in sorted(Path(root).rglob("*")):
        if path.is_file():
            digest = hashlib.sha256()
            with path.open("rb") as handle:
                while chunk := handle.read(1024 * 1024):
                    digest.update(chunk)
            hashes[str(path.relative_to(root))] = digest.hexdigest()
    return hashes


## 2. Problem frame — what decision will this data support?

The product goal is not “find unusual numbers.” It is to rank operational incidents
that deserve attention. The statements below are hypotheses and constraints to test,
not facts imported from a domain expert.


In [ ]:
problem_frame = pd.DataFrame([
    {
        "question": "What is ranked?",
        "telecom_working_hypothesis": "entity or shared-domain incident episode",
        "petrobras_3w_working_hypothesis": "well-instance condition episode",
        "epistemic_status": "product hypothesis",
        "must_validate_with": "operator or domain practitioner",
    },
    {
        "question": "What evidence may the detector use at scoring time?",
        "telecom_working_hypothesis": "telemetry, topology, service validity, known operational events",
        "petrobras_3w_working_hypothesis": "sensor telemetry and known well identity",
        "epistemic_status": "design constraint",
        "must_validate_with": "data owner and deployment design",
    },
    {
        "question": "What is success?",
        "telecom_working_hypothesis": "useful incidents near the top at tolerable daily alert volume",
        "petrobras_3w_working_hypothesis": "useful condition episodes near the top; not row classification alone",
        "epistemic_status": "product hypothesis",
        "must_validate_with": "operator workflow",
    },
    {
        "question": "Can source prevalence estimate deployment alert volume?",
        "telecom_working_hypothesis": "no — fixture prevalence is generator-controlled",
        "petrobras_3w_working_hypothesis": "no — files are event-conditioned",
        "epistemic_status": "sampling fact",
        "must_validate_with": "deployment data",
    },
])
write_csv(problem_frame, "problem_frame.csv")
display(problem_frame)


## 3. Exact source inventory — metadata, not a sample


In [ ]:
if EDA_SECTOR == "telecom":
    discovery = discover_telecom(SOURCE_ROOT)
    assert discovery["core_ready"], discovery
    panel_path = native_path(SOURCE_ROOT, "reference_dataset.parquet")
    panel_parquet = pq.ParquetFile(panel_path)
    topology = pd.read_csv(native_path(SOURCE_ROOT, "topology.csv"))
    service_windows = pd.read_csv(
        native_path(SOURCE_ROOT, "entity_service_windows.csv")
    )
    exact_start, exact_end = timestamp_bounds(panel_parquet, "timestamp_utc")
    inventory_specs = [
        ("reference_dataset.parquet", "operational telemetry", True),
        ("topology.csv", "operational entity and peer context", True),
        ("entity_service_windows.csv", "operational validity", True),
        ("engineering_events.csv", "operational context", False),
    ]
    rows = []
    for filename, role, required in inventory_specs:
        path = native_path(SOURCE_ROOT, filename, required=False)
        if path is None:
            rows.append({"file": filename, "role": role, "present": False})
            continue
        shape = parquet_shape(path) if path.suffix == ".parquet" else csv_shape(path)
        rows.append({
            "file": filename,
            "role": role,
            "present": True,
            "rows": shape[0],
            "columns": shape[1],
            "size_mb": path.stat().st_size / 1024**2,
        })
    source_inventory = pd.DataFrame(rows)
    exact_file_count = int(source_inventory["present"].sum())
    exact_row_count = int(panel_parquet.metadata.num_rows)
    exact_column_count = len(panel_parquet.schema_arrow.names)
    exact_entity_count = int(topology["ont_id"].astype(str).nunique())
    file_inventory = pd.DataFrame()
else:
    discovery = discover_threew(SOURCE_ROOT)
    assert discovery["ready"], discovery
    paths = sorted(
        path
        for directory in range(10)
        for path in (SOURCE_ROOT / str(directory)).glob("*.parquet")
    )
    inventory_rows = []
    for position, path in enumerate(paths, start=1):
        parquet = pq.ParquetFile(path)
        start, end = timestamp_bounds(parquet, "timestamp")
        kind = threew_source_kind(path)
        entity_id = path.stem.split("_", 1)[0] if kind == "real" else path.stem
        inventory_rows.append({
            "file_id": opaque_id(path.stem, "file"),
            "file_key": path.stem,
            "source_kind": kind,
            "entity_id": entity_id,
            "rows": parquet.metadata.num_rows,
            "columns": len(parquet.schema_arrow.names),
            "start_ts": start,
            "end_ts": end,
            "size_mb": path.stat().st_size / 1024**2,
            "path": path,
        })
        if position % 400 == 0:
            print(f"Scanned metadata for {position:,}/{len(paths):,} files")
    file_inventory = pd.DataFrame(inventory_rows)
    source_inventory = (
        file_inventory.groupby("source_kind", as_index=False)
        .agg(
            files=("file_id", "size"),
            rows=("rows", "sum"),
            entities=("entity_id", "nunique"),
            size_mb=("size_mb", "sum"),
        )
    )
    exact_file_count = len(file_inventory)
    exact_row_count = int(file_inventory["rows"].sum())
    exact_column_count = int(file_inventory["columns"].max())
    exact_entity_count = int(
        file_inventory.loc[
            file_inventory["source_kind"].eq("real"), "entity_id"
        ].nunique()
    )
    exact_start = file_inventory["start_ts"].min()
    exact_end = file_inventory["end_ts"].max()
    panel_path = None
    panel_parquet = None

write_csv(source_inventory, "source_inventory.csv")
display(source_inventory)
display(pd.Series({
    "exact_files": exact_file_count,
    "exact_rows": exact_row_count,
    "maximum_columns_per_record": exact_column_count,
    "primary_real_entities": exact_entity_count,
    "metadata_time_start": exact_start,
    "metadata_time_end": exact_end,
}, name="exact_value").to_frame())


## 4. Seal the exploration frame before feature analysis

**Telecom:** hash 70% of ONTs into development and keep 30% as a cold-entity
holdout. Only the first 60% of development time is explored; the later 40% is a
warm-future holdout.

**3W:** hash real well IDs into development/cold-well partitions, then hash files
within development wells into exploration/unseen-instance partitions. A global time
split would be invalid because event files are separate trajectories.

The holdouts are sealed here and not analysed in this notebook.


In [ ]:
if EDA_SECTOR == "telecom":
    all_entities = sorted(topology["ont_id"].dropna().astype(str).unique())
    entity_partition = pd.DataFrame({
        "entity_id": all_entities,
        "partition": [
            "development" if hash_fraction(value, "entity") < DEV_ENTITY_FRACTION
            else "cold_entity_holdout"
            for value in all_entities
        ],
    })
    development_entities = set(
        entity_partition.loc[
            entity_partition["partition"].eq("development"), "entity_id"
        ]
    )
    exploration_cutoff = exact_start + (exact_end - exact_start) * DEV_TIME_FRACTION
    entity_partition["exploration_start"] = exact_start
    entity_partition["exploration_cutoff"] = exploration_cutoff
    entity_partition["holdout_end"] = exact_end

    row_group_eligible = []
    partition_counts = {
        "development_exploration": 0,
        "development_future_holdout": 0,
        "cold_entity_holdout": 0,
    }
    full_entity_counts = {}
    full_month_counts = {}
    for row_group in range(panel_parquet.num_row_groups):
        keys = panel_parquet.read_row_group(
            row_group, columns=["timestamp_utc", "ont_id"]
        ).to_pandas()
        entities = keys["ont_id"].astype(str)
        timestamps = pd.to_datetime(keys["timestamp_utc"], utc=True)
        is_dev = entities.isin(development_entities)
        is_explore = is_dev & timestamps.lt(exploration_cutoff)
        row_group_eligible.append(int(is_explore.sum()))
        partition_counts["development_exploration"] += int(is_explore.sum())
        partition_counts["development_future_holdout"] += int(
            (is_dev & ~is_explore).sum()
        )
        partition_counts["cold_entity_holdout"] += int((~is_dev).sum())
        explored = pd.DataFrame({
            "entity_id": entities.loc[is_explore],
            "month": timestamps.loc[is_explore].dt.to_period("M").astype(str),
        })
        for key, value in explored["entity_id"].value_counts().items():
            full_entity_counts[key] = full_entity_counts.get(key, 0) + int(value)
        for key, value in explored["month"].value_counts().items():
            full_month_counts[key] = full_month_counts.get(key, 0) + int(value)

    seal_table = entity_partition
    primary_partition = "development entities before exploration cutoff"
else:
    real_inventory = file_inventory.loc[
        file_inventory["source_kind"].eq("real")
    ].copy()
    real_inventory["entity_partition"] = real_inventory["entity_id"].map(
        lambda value: (
            "development"
            if hash_fraction(value, "entity") < DEV_ENTITY_FRACTION
            else "cold_well_holdout"
        )
    )
    real_inventory["partition"] = "cold_well_holdout"
    is_dev = real_inventory["entity_partition"].eq("development")
    real_inventory.loc[is_dev, "partition"] = real_inventory.loc[
        is_dev, "file_key"
    ].map(
        lambda value: (
            "development_exploration"
            if hash_fraction(value, "instance") < THREEW_DEV_FILE_FRACTION
            else "development_unseen_instance_holdout"
        )
    )
    exploration_inventory = real_inventory.loc[
        real_inventory["partition"].eq("development_exploration")
    ].copy()
    partition_counts = (
        real_inventory.groupby("partition")["rows"].sum().astype(int).to_dict()
    )
    seal_table = real_inventory[
        ["file_id", "entity_id", "partition", "rows", "start_ts", "end_ts"]
    ].copy()
    exploration_cutoff = None
    primary_partition = "real WELL files in development_exploration"

write_csv(seal_table, "holdout_seal.csv")
analysis_frame_manifest = {
    "sector": EDA_SECTOR,
    "seed": RANDOM_SEED,
    "entity_hash_rule": f"development when SHA256 fraction < {DEV_ENTITY_FRACTION}",
    "telecom_time_rule": (
        f"timestamp < {exploration_cutoff}" if EDA_SECTOR == "telecom" else None
    ),
    "threew_file_rule": (
        f"development file exploration when SHA256 fraction < "
        f"{THREEW_DEV_FILE_FRACTION}"
        if EDA_SECTOR == "petrobras_3w" else None
    ),
    "primary_exploration_partition": primary_partition,
    "partition_row_counts": partition_counts,
    "holdouts_are_not_analysed_here": True,
}
write_json(analysis_frame_manifest, "analysis_frame_manifest.json")
display(pd.Series(partition_counts, name="rows").to_frame())
display(seal_table.head(10))


## 5. Feature dictionary — print and understand every field

`role` is the leakage boundary. `measurement_kind` controls valid statistical
treatment. `nuisance_modes` lists non-fault mechanisms that could create surprising
values. A hypothesis is deliberately labelled as a hypothesis; it is not promoted to
domain fact.


In [ ]:
TELECOM_METRIC_INFO = {
    "rx_power_dbm": ("received optical power", "dBm", "continuous gauge", "low", "attenuation, distance, shared optical path, clipping"),
    "tx_power_dbm": ("transmitted optical power", "dBm", "continuous gauge", "both", "laser control, vendor calibration, temperature"),
    "temperature_c": ("ONT temperature", "degC", "continuous gauge", "high", "ambient cycle, enclosure, seasonal drift"),
    "bias_current_ma": ("laser bias current", "mA", "continuous gauge", "high", "temperature, laser ageing, vendor calibration"),
    "voltage_v": ("ONT supply voltage", "V", "continuous gauge", "both", "power supply, measurement quantisation"),
    "ber": ("bit error ratio", "ratio", "bounded fraction", "high", "zero inflation, detection limit, optical impairment"),
    "fec_count": ("generator FEC-related interval count", "count", "interval count", "high", "upper censoring at 5e6, exposure, vendor scale"),
    "crc_errors": ("CRC error interval count", "count", "interval count", "high", "zero inflation, traffic exposure"),
    "uptime_s": ("seconds since last restart", "s", "cumulative counter", "change", "legitimate resets, wrap or collection reset"),
    "reboot_count": ("cumulative reboot count", "count", "cumulative counter", "high", "legitimate increments, reset or wrap"),
    "throughput_mbps": ("service throughput", "Mbps", "continuous gauge", "low", "traffic demand, time of day, service plan"),
}
TELECOM_CONTEXT_INFO = {
    "timestamp_utc": ("observation time", "timestamp"),
    "ont_id": ("optical network terminal identifier", "identifier"),
    "olt_id": ("optical line terminal peer group", "topology context"),
    "pon_port": ("passive optical network port", "topology context"),
    "splitter_l1": ("first-level optical splitter", "topology context"),
    "splitter_l2": ("second-level optical splitter", "topology context"),
    "geo_cluster": ("geographic peer group", "non-tree context"),
    "device_model": ("ONT device model", "entity context"),
    "vendor": ("ONT vendor", "entity context"),
    "enclosure": ("installation enclosure", "entity context"),
    "firmware_version": ("firmware version", "entity context"),
    "distance_m": ("optical path distance", "entity context"),
    "distance_bucket": ("binned optical distance", "entity context"),
    "splitter_ratio": ("split ratio", "entity context"),
    "l2_splitter_capacity": ("second-level splitter capacity", "entity context"),
    "fibre_age_yr": ("fibre age", "entity context"),
    "expected_rx_power_dbm": ("engineering expected receive power", "entity context"),
    "rx_sensitivity_dbm": ("receiver sensitivity", "entity context"),
    "service_impact_weight": ("synthetic service-impact weight", "entity context"),
    "customer_priority_weight": ("synthetic customer-priority weight", "entity context"),
}

if EDA_SECTOR == "telecom":
    native_schema = pd.DataFrame([
        {"field": field.name, "physical_type": str(field.type)}
        for field in panel_parquet.schema_arrow
    ])
    feature_rows = []
    for record in native_schema.itertuples(index=False):
        field = record.field
        if field in TELECOM_METRIC_INFO:
            meaning, unit, kind, direction, nuisance = TELECOM_METRIC_INFO[field]
            role = "operational_measurement"
            availability = "available_at_scoring_time"
            evidence = "pack definition; nuisance list partly hypothesis"
        elif field.startswith("gt_"):
            meaning, unit, kind, direction, nuisance = (
                "synthetic evaluation truth; quarantined",
                None, "evaluation label", None, "not an input feature",
            )
            role = "evaluation_only"
            availability = "forbidden_at_scoring_time"
            evidence = "native naming and generator documentation"
        else:
            meaning, context_kind = TELECOM_CONTEXT_INFO.get(
                field, (field.replace("_", " "), "context")
            )
            unit = "m" if field == "distance_m" else (
                "yr" if field == "fibre_age_yr" else None
            )
            kind, direction = context_kind, None
            nuisance = "cohort composition; do not treat identifiers as magnitudes"
            role = "timestamp_or_context"
            availability = "available_at_scoring_time"
            evidence = "native schema and pack definition"
        feature_rows.append({
            "native_field": field,
            "meaning": meaning,
            "unit": unit,
            "physical_type": record.physical_type,
            "measurement_kind": kind,
            "role": role,
            "inference_availability": availability,
            "anomaly_direction": direction,
            "initial_treatment": (
                "quarantine" if role == "evaluation_only"
                else "transform by measurement kind; retain nulls"
            ),
            "nuisance_modes": nuisance,
            "epistemic_status": evidence,
        })
    timestamp_field, entity_field = "timestamp_utc", "ont_id"
    metric_fields = list(TELECOM_METRIC_INFO)
    context_fields = [
        field for field in TELECOM_CONTEXT_INFO
        if field not in {timestamp_field, entity_field}
        and field in native_schema["field"].tolist()
    ]
    truth_fields = [
        field for field in native_schema["field"] if field.startswith("gt_")
    ]
    grain = "one ONT poll at one timestamp"
else:
    first_real_path = real_inventory.iloc[0]["path"]
    first_parquet = pq.ParquetFile(first_real_path)
    native_schema = pd.DataFrame([
        {"field": field.name, "physical_type": str(field.type)}
        for field in first_parquet.schema_arrow
    ])
    parser = configparser.ConfigParser()
    parser.optionxform = str
    parser.read(SOURCE_ROOT / "dataset.ini")
    descriptions = dict(parser["PARQUET_FILE_PROPERTIES"])
    feature_rows = []
    for record in native_schema.itertuples(index=False):
        field = record.field
        description = descriptions.get(field, field)
        unit_match = re.search(r"\[([^\]]+)\]\s*$", description)
        unit = unit_match.group(1) if unit_match else None
        meaning = re.sub(r"\s*\[[^\]]+\]\s*$", "", description)
        if field in {"class", "state"}:
            role, kind = "evaluation_only", "evaluation label"
            availability = "forbidden_at_scoring_time"
            nuisance = "not an input feature"
            treatment = "quarantine"
        elif field == "timestamp":
            role, kind = "timestamp_or_context", "timestamp"
            availability = "available_at_scoring_time"
            nuisance = "file-local time origin and variable duration"
            treatment = "ordering and cadence only"
        elif field.startswith("ESTADO-"):
            role, kind = "operational_measurement", "discrete state"
            availability = "available_at_scoring_time"
            nuisance = "legitimate valve operations, frozen state, missing state"
            treatment = "transitions and dwell times; not Gaussian scaling"
        elif field.startswith("ABER-"):
            role, kind = "operational_measurement", "bounded gauge"
            availability = "available_at_scoring_time"
            nuisance = "control action, saturation at bounds, frozen actuator"
            treatment = "retain bounds; inspect transitions"
        else:
            role, kind = "operational_measurement", "continuous gauge"
            availability = "available_at_scoring_time"
            nuisance = "operating regime, sensor dropout, frozen sensor, calibration"
            treatment = "robust scaling within series and cohort"
        feature_rows.append({
            "native_field": field,
            "meaning": meaning,
            "unit": unit,
            "physical_type": record.physical_type,
            "measurement_kind": kind,
            "role": role,
            "inference_availability": availability,
            "anomaly_direction": "both" if role == "operational_measurement" else None,
            "initial_treatment": treatment,
            "nuisance_modes": nuisance,
            "epistemic_status": (
                "dataset.ini description; nuisance modes are working hypotheses"
            ),
        })
    timestamp_field, entity_field = "timestamp", "entity_id"
    metric_fields = [
        row["native_field"] for row in feature_rows
        if row["role"] == "operational_measurement"
    ]
    context_fields = []
    truth_fields = ["class", "state", "event_directory"]
    grain = "one timestamp within one well-event instance file"

feature_dictionary = pd.DataFrame(feature_rows)
write_csv(feature_dictionary, "feature_dictionary.csv")
write_csv(native_schema, "native_schema.csv")
display(pd.Series({
    "row_grain": grain,
    "operational_feature_count": len(metric_fields),
    "evaluation_fields_quarantined": ", ".join(truth_fields),
}, name="value").to_frame())
display(feature_dictionary)


## 6. Draw a reproducible operational sample

This sample is for distributions and cross-sectional summaries. It contains no truth
values and only comes from the sealed exploration partition. The complete histories
used later are loaded separately; a random row sample is not suitable for temporal
analysis.


In [ ]:
if EDA_SECTOR == "telecom":
    read_columns = list(dict.fromkeys([
        timestamp_field, entity_field, *metric_fields, *context_fields
    ]))
    allocations = allocate_integer(SAMPLE_ROWS, row_group_eligible)
    sample_parts, manifest_rows = [], []
    for row_group, requested in enumerate(allocations):
        if requested == 0:
            continue
        frame = panel_parquet.read_row_group(
            row_group, columns=read_columns
        ).to_pandas()
        timestamps = pd.to_datetime(frame[timestamp_field], utc=True)
        eligible = (
            frame[entity_field].astype(str).isin(development_entities)
            & timestamps.lt(exploration_cutoff)
        )
        eligible_positions = np.flatnonzero(eligible.to_numpy())
        take = min(len(eligible_positions), int(requested))
        local_rng = np.random.default_rng(RANDOM_SEED + row_group)
        positions = np.sort(
            local_rng.choice(eligible_positions, size=take, replace=False)
        )
        selected = frame.iloc[positions].copy()
        selected["_row_group"] = row_group
        selected["_native_position"] = positions
        sample_parts.append(selected)
        manifest_rows.append({
            "source_unit": f"row_group_{row_group}",
            "source_rows_in_exploration": len(eligible_positions),
            "sample_rows": take,
        })
    analysis_sample = pd.concat(sample_parts, ignore_index=True)
    sample_manifest = pd.DataFrame(manifest_rows)
    sample_design = "proportional random sample across all exploration row groups"
else:
    ordered = exploration_inventory.assign(
        _selection_hash=exploration_inventory["file_key"].map(
            lambda value: hash_fraction(value, "sample_file")
        )
    ).sort_values("_selection_hash")
    selected_files = ordered.head(min(THREEW_FILE_COUNT, len(ordered))).copy()
    sample_parts, manifest_rows = [], []
    for position, record in enumerate(selected_files.itertuples(index=False)):
        frame = read_threew(record.path, [timestamp_field, *metric_fields])
        take = min(len(frame), THREEW_ROWS_PER_FILE)
        local_rng = np.random.default_rng(RANDOM_SEED + position)
        native_positions = np.sort(
            local_rng.choice(len(frame), size=take, replace=False)
        )
        selected = frame.iloc[native_positions].copy()
        selected[entity_field] = record.entity_id
        selected["source_file_id"] = record.file_id
        selected["_native_position"] = native_positions
        sample_parts.append(selected)
        manifest_rows.append({
            "source_file_id": record.file_id,
            "entity_id": record.entity_id,
            "source_rows": len(frame),
            "sample_rows": take,
            "duration_hours": (
                (record.end_ts - record.start_ts).total_seconds() / 3600
                if pd.notna(record.start_ts) and pd.notna(record.end_ts) else None
            ),
        })
    analysis_sample = pd.concat(sample_parts, ignore_index=True)
    sample_manifest = pd.DataFrame(manifest_rows)
    sample_design = (
        "uniform deterministic file sample from real-WELL exploration files; "
        "equal bounded rows per selected file"
    )

analysis_sample[timestamp_field] = pd.to_datetime(
    analysis_sample[timestamp_field], utc=True, errors="coerce"
)
operational_columns = list(dict.fromkeys([
    timestamp_field, entity_field, *metric_fields, *context_fields
]))
operational_sample = analysis_sample[operational_columns].copy()
sample_duplicate_keys = int(
    analysis_sample.duplicated(
        [entity_field, timestamp_field]
        if EDA_SECTOR == "telecom"
        else ["source_file_id", timestamp_field]
    ).sum()
)
write_csv(sample_manifest, "sample_manifest.csv")
display(pd.Series({
    "sample_design": sample_design,
    "rows": len(operational_sample),
    "entities": operational_sample[entity_field].nunique(),
    "time_start": operational_sample[timestamp_field].min(),
    "time_end": operational_sample[timestamp_field].max(),
    "duplicate_grain_keys": sample_duplicate_keys,
}, name="sample_value").to_frame())
display(sample_manifest.head(20))


## 7. See the data before summarising it

The first table shows the first sampled records in time order, the second shows a
deterministic random set, and the third transposes one observation so every field is
readable. These are operational columns only.


In [ ]:
requested_focus = [
    value.strip()
    for value in os.getenv("EDA_FOCUS_FEATURES", "").split(",")
    if value.strip()
]
default_focus = (
    ["rx_power_dbm", "temperature_c", "ber", "fec_count", "crc_errors", "throughput_mbps"]
    if EDA_SECTOR == "telecom"
    else ["P-ANULAR", "P-MON-CKP", "P-TPT", "QGL", "T-TPT", "ABER-CKP", "ESTADO-DHSV", "ESTADO-M1"]
)
focus_metrics = [
    field for field in (requested_focus or default_focus) if field in metric_fields
]

preview_columns = list(dict.fromkeys([
    timestamp_field, entity_field, *focus_metrics,
    *context_fields[:5],
]))
print("First operational rows in the exploration sample:")
display(
    operational_sample[preview_columns]
    .sort_values([entity_field, timestamp_field])
    .head(10)
)
print("Deterministic random operational rows:")
display(
    operational_sample[preview_columns]
    .sample(n=min(10, len(operational_sample)), random_state=RANDOM_SEED)
    .sort_values(timestamp_field)
)
print("One observation, transposed:")
display(operational_sample[preview_columns].iloc[[0]].T.rename(columns={0: "value"}))
print("Meaning and treatment of the focus features:")
display(
    feature_dictionary.loc[
        feature_dictionary["native_field"].isin(focus_metrics),
        [
            "native_field", "meaning", "unit", "measurement_kind",
            "initial_treatment", "nuisance_modes", "epistemic_status",
        ],
    ]
)


## 8. Is the bounded sample representative of its exploration frame?


In [ ]:
sampling_rows = []
if EDA_SECTOR == "telecom":
    sample_entity_counts = operational_sample[entity_field].astype(str).value_counts()
    full_entities = pd.Series(full_entity_counts, dtype=float)
    full_entity_share = full_entities / full_entities.sum()
    sample_entity_share = sample_entity_counts / sample_entity_counts.sum()
    aligned = pd.concat(
        [full_entity_share.rename("full"), sample_entity_share.rename("sample")],
        axis=1,
    ).fillna(0)
    sample_months = (
        operational_sample[timestamp_field].dt.to_period("M").astype(str)
        .value_counts()
    )
    full_month_share = pd.Series(full_month_counts, dtype=float)
    full_month_share /= full_month_share.sum()
    sample_month_share = sample_months / sample_months.sum()
    month_aligned = pd.concat(
        [full_month_share.rename("full"), sample_month_share.rename("sample")],
        axis=1,
    ).fillna(0)
    sampling_rows.extend([
        {
            "check": "development entity coverage",
            "estimate": sample_entity_counts.index.nunique() / len(development_entities),
            "criterion": ">= 0.90",
            "status": "pass" if sample_entity_counts.index.nunique() / len(development_entities) >= 0.90 else "review",
        },
        {
            "check": "maximum absolute entity-share difference",
            "estimate": float((aligned["full"] - aligned["sample"]).abs().max()),
            "criterion": "<= 0.01",
            "status": "pass" if (aligned["full"] - aligned["sample"]).abs().max() <= 0.01 else "review",
        },
        {
            "check": "maximum absolute month-share difference",
            "estimate": float((month_aligned["full"] - month_aligned["sample"]).abs().max()),
            "criterion": "<= 0.03",
            "status": "pass" if (month_aligned["full"] - month_aligned["sample"]).abs().max() <= 0.03 else "review",
        },
    ])
else:
    full_duration = (
        exploration_inventory["end_ts"] - exploration_inventory["start_ts"]
    ).dt.total_seconds() / 3600
    selected_duration = pd.to_numeric(sample_manifest["duration_hours"], errors="coerce")
    sampling_rows.extend([
        {
            "check": "selected real exploration files",
            "estimate": len(sample_manifest),
            "criterion": f"{min(THREEW_FILE_COUNT, len(exploration_inventory))}",
            "status": "pass",
        },
        {
            "check": "selected well coverage",
            "estimate": sample_manifest["entity_id"].nunique(),
            "criterion": "at least 3 wells",
            "status": "pass" if sample_manifest["entity_id"].nunique() >= 3 else "review",
        },
        {
            "check": "selected/full median duration ratio",
            "estimate": float(selected_duration.median() / full_duration.median()),
            "criterion": "0.5 to 2.0",
            "status": (
                "pass"
                if 0.5 <= selected_duration.median() / full_duration.median() <= 2.0
                else "review"
            ),
        },
    ])

sampling_validity = pd.DataFrame(sampling_rows)
write_csv(sampling_validity, "sampling_validity.csv")
display(sampling_validity)


## 9. Univariate structure, robust shape and data-quality clues

Moment skew can be dominated by a few extremes. `bowley_skew` and `medcouple` are
robust shape summaries. Tail rows are **flagged, not deleted**. Histograms winsorise
only the displayed axis; reported statistics use the original observed values.


In [ ]:
def robust_numeric_summary(frame, fields):
    rows = []
    for field in fields:
        values = pd.to_numeric(frame[field], errors="coerce")
        finite = values[np.isfinite(values)].dropna()
        q = finite.quantile([0.01, 0.25, 0.50, 0.75, 0.99])
        iqr = q.get(0.75, np.nan) - q.get(0.25, np.nan)
        bowley = (
            (q.get(0.75) + q.get(0.25) - 2 * q.get(0.50)) / iqr
            if len(finite) and iqr > 0 else np.nan
        )
        if len(finite) > 5000:
            finite_for_shape = finite.sample(5000, random_state=RANDOM_SEED)
        else:
            finite_for_shape = finite
        robust_skew = (
            float(medcouple(finite_for_shape.to_numpy()))
            if len(finite_for_shape) >= 10
            and finite_for_shape.nunique() > 1
            else np.nan
        )
        rows.append({
            "field": field,
            "sample_rows": len(values),
            "valid_count": len(finite),
            "missing_fraction": values.isna().mean(),
            "infinite_count": int((values.notna() & ~np.isfinite(values)).sum()),
            "unique_values": finite.nunique(),
            "zero_fraction_of_valid": finite.eq(0).mean() if len(finite) else np.nan,
            "min": finite.min() if len(finite) else np.nan,
            "p01": q.get(0.01, np.nan),
            "p25": q.get(0.25, np.nan),
            "median": q.get(0.50, np.nan),
            "p75": q.get(0.75, np.nan),
            "p99": q.get(0.99, np.nan),
            "max": finite.max() if len(finite) else np.nan,
            "mean": finite.mean() if len(finite) else np.nan,
            "std": finite.std() if len(finite) else np.nan,
            "bowley_skew": bowley,
            "medcouple": robust_skew,
            "sample_min_fraction": finite.eq(finite.min()).mean() if len(finite) else np.nan,
            "sample_max_fraction": finite.eq(finite.max()).mean() if len(finite) else np.nan,
        })
    return pd.DataFrame(rows)


numeric_summary = robust_numeric_summary(operational_sample, metric_fields)
write_csv(numeric_summary, "numeric_summary_sample.csv")
display(numeric_summary)


In [ ]:
quality_rows = []
for record in numeric_summary.itertuples(index=False):
    checks = [
        (record.missing_fraction >= 0.05, "material_missingness", record.missing_fraction),
        (record.zero_fraction_of_valid >= 0.50, "zero_inflated", record.zero_fraction_of_valid),
        (record.unique_values <= 3, "discrete_or_low_cardinality", record.unique_values),
        (record.sample_max_fraction >= 0.10, "upper_boundary_concentration", record.sample_max_fraction),
        (abs(record.medcouple) >= 0.50 if pd.notna(record.medcouple) else False, "robust_extreme_skew", record.medcouple),
        (record.infinite_count > 0, "infinite_values", record.infinite_count),
    ]
    for triggered, issue, evidence in checks:
        if triggered:
            quality_rows.append({
                "field": record.field,
                "issue_to_investigate": issue,
                "sample_evidence": evidence,
                "action": "retain values; encode semantics before modelling",
            })

quality_flags = pd.DataFrame(
    quality_rows,
    columns=["field", "issue_to_investigate", "sample_evidence", "action"],
)
write_csv(quality_flags, "quality_flags_sample.csv")
display(quality_flags)


In [ ]:
entity_missingness = (
    operational_sample.groupby(entity_field, observed=True)[metric_fields]
    .agg(lambda values: values.isna().mean())
)
entity_missingness_summary = pd.DataFrame([
    {
        "field": field,
        "entity_median_missing_fraction": entity_missingness[field].median(),
        "entity_p95_missing_fraction": entity_missingness[field].quantile(0.95),
        "entity_max_missing_fraction": entity_missingness[field].max(),
        "entities_fully_missing": int(entity_missingness[field].eq(1).sum()),
    }
    for field in metric_fields
])
missing_indicator = operational_sample[focus_metrics].isna().astype(float)
missingness_cooccurrence = missing_indicator.corr(method="pearson")
write_csv(entity_missingness_summary, "entity_missingness_summary.csv")
write_csv(missingness_cooccurrence, "missingness_cooccurrence.csv", index=True)
display(entity_missingness_summary)

figure, axes = plt.subplots(1, 2, figsize=(15, max(5, len(focus_metrics) * 0.45)))
missing_order = numeric_summary.sort_values("missing_fraction")
axes[0].barh(missing_order["field"], missing_order["missing_fraction"], color="#4C78A8")
axes[0].set_title("Global sample missingness")
axes[0].set_xlabel("missing fraction")
axes[0].grid(axis="x", alpha=0.25)
image = axes[1].imshow(missingness_cooccurrence, vmin=-1, vmax=1, cmap="coolwarm")
axes[1].set_xticks(range(len(focus_metrics)))
axes[1].set_yticks(range(len(focus_metrics)))
axes[1].set_xticklabels(focus_metrics, rotation=90)
axes[1].set_yticklabels(focus_metrics)
axes[1].set_title("Co-occurrence of missing indicators")
figure.colorbar(image, ax=axes[1], fraction=0.046)
save_figure(figure, "01_missingness.png")


In [ ]:
plot_metrics = focus_metrics[:12]
columns = 3
rows = math.ceil(len(plot_metrics) / columns)
figure, axes = plt.subplots(
    rows, columns, figsize=(15, 3.6 * rows), squeeze=False
)
for axis, field in zip(axes.flat, plot_metrics):
    values = pd.to_numeric(
        operational_sample[field], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan).dropna()
    if values.nunique() <= 1:
        axis.text(0.5, 0.5, "constant or unavailable", ha="center")
        axis.set_title(field)
        continue
    lower, upper = values.quantile([0.005, 0.995])
    axis.hist(values.clip(lower, upper), bins=50, color="#4C78A8", alpha=0.85)
    axis.set_title(field)
    axis.set_xlabel("bulk view: sample 0.5%–99.5%")
    axis.set_ylabel("sample rows")
    axis.grid(alpha=0.2)
for axis in axes.flat[len(plot_metrics):]:
    axis.axis("off")
figure.suptitle(f"{EDA_SECTOR}: operational feature distributions", y=1.01)
save_figure(figure, "02_distributions.png")


## 10. Load complete selected histories

Temporal diagnostics must preserve order, gaps, and state transitions. Telecom selects
development ONTs to include peers from shared splitters. 3W loads complete trajectories
from a small deterministic set of real exploration files. No selected history is
truncated for calculation; only plotting is downsampled.


In [ ]:
if EDA_SECTOR == "telecom":
    topology_work = topology.copy()
    topology_work["ont_id"] = topology_work["ont_id"].astype(str)
    topology_work = topology_work.loc[
        topology_work["ont_id"].isin(development_entities)
    ].copy()
    topology_work["_hash"] = topology_work["ont_id"].map(
        lambda value: hash_fraction(value, "longitudinal")
    )
    peer_candidates = (
        topology_work.sort_values(["splitter_l2", "_hash"])
        .groupby("splitter_l2", sort=True)
        .head(3)
    )
    selected_entity_table = peer_candidates.head(
        LONGITUDINAL_ENTITY_COUNT
    ).copy()
    if len(selected_entity_table) < LONGITUDINAL_ENTITY_COUNT:
        remaining = topology_work.loc[
            ~topology_work["ont_id"].isin(selected_entity_table["ont_id"])
        ].sort_values("_hash")
        selected_entity_table = pd.concat([
            selected_entity_table,
            remaining.head(LONGITUDINAL_ENTITY_COUNT - len(selected_entity_table)),
        ])
    selected_entities = set(selected_entity_table["ont_id"])
    series_metrics = focus_metrics
    series_parts = []
    for batch in panel_parquet.iter_batches(
        batch_size=100000,
        columns=[timestamp_field, entity_field, *series_metrics],
    ):
        frame = batch.to_pandas()
        timestamps = pd.to_datetime(frame[timestamp_field], utc=True)
        selected = frame.loc[
            frame[entity_field].astype(str).isin(selected_entities)
            & timestamps.lt(exploration_cutoff)
        ].copy()
        if len(selected):
            series_parts.append(selected)
    series_data = pd.concat(series_parts, ignore_index=True)
    series_data["series_id"] = series_data[entity_field].astype(str)
    longitudinal_manifest = selected_entity_table[
        ["ont_id", "olt_id", "pon_port", "splitter_l1", "splitter_l2", "geo_cluster"]
    ].rename(columns={"ont_id": "series_id"})
else:
    ordered = exploration_inventory.assign(
        _long_hash=exploration_inventory["file_key"].map(
            lambda value: hash_fraction(value, "longitudinal")
        )
    ).sort_values("_long_hash")
    longitudinal_files = ordered.head(
        min(LONGITUDINAL_FILE_COUNT, len(ordered))
    ).copy()
    series_metrics = focus_metrics
    series_parts = []
    for record in longitudinal_files.itertuples(index=False):
        frame = read_threew(record.path, [timestamp_field, *series_metrics])
        frame[entity_field] = record.entity_id
        frame["series_id"] = record.file_id
        series_parts.append(frame)
    series_data = pd.concat(series_parts, ignore_index=True)
    longitudinal_manifest = longitudinal_files[
        ["file_id", "entity_id", "rows", "start_ts", "end_ts"]
    ].rename(columns={"file_id": "series_id"})

series_data[timestamp_field] = pd.to_datetime(
    series_data[timestamp_field], utc=True, errors="coerce"
)
series_data = series_data.sort_values(
    ["series_id", timestamp_field]
).reset_index(drop=True)
write_csv(longitudinal_manifest, "longitudinal_manifest.csv")
display(longitudinal_manifest)
print(
    f"Loaded {len(series_data):,} rows from "
    f"{series_data['series_id'].nunique()} complete selected histories."
)
display(series_data[[timestamp_field, entity_field, "series_id", *series_metrics]].head(10))


In [ ]:
plot_series = list(series_data["series_id"].drop_duplicates())[:SERIES_PLOT_COUNT]
figure, axes = plt.subplots(
    len(plot_series), len(series_metrics),
    figsize=(4.2 * len(series_metrics), 3.0 * len(plot_series)),
    squeeze=False,
)
for row_index, series_id in enumerate(plot_series):
    frame = series_data.loc[series_data["series_id"].eq(series_id)]
    if len(frame) > SERIES_MAX_POINTS:
        positions = np.linspace(0, len(frame) - 1, SERIES_MAX_POINTS, dtype=int)
        frame = frame.iloc[positions]
    for column_index, field in enumerate(series_metrics):
        axis = axes[row_index, column_index]
        plotted_values = pd.to_numeric(frame[field], errors="coerce")
        if plotted_values.notna().any():
            axis.plot(
                frame[timestamp_field],
                plotted_values,
                linewidth=0.75,
            )
            axis.tick_params(axis="x", rotation=30)
        else:
            axis.text(
                0.5, 0.5, "no observed values", ha="center", va="center",
                transform=axis.transAxes,
            )
            axis.set_xticks([])
        axis.set_title(f"{series_id}\n{field}", fontsize=8)
        axis.grid(alpha=0.2)
figure.suptitle("Complete selected histories (display downsampled only)", y=1.01)
save_figure(figure, "03_complete_series.png")


## 11. Cadence, gaps, frozen measurements and effective sample size

`n_eff` is an autocorrelation-adjusted information estimate, not a new row count.
Lag correlations require exact timestamp separation, so a gap is never silently
bridged. Low pair coverage is reported.


In [ ]:
def exact_lag_acf(timestamps, values, cadence, max_lag=48):
    times = pd.to_datetime(timestamps, utc=True).astype("int64").to_numpy()
    values = pd.to_numeric(values, errors="coerce").to_numpy(dtype=float)
    cadence_ns = int(cadence.value)
    rows = []
    for lag in range(1, max_lag + 1):
        if len(values) <= lag:
            break
        valid = (
            np.isfinite(values[lag:])
            & np.isfinite(values[:-lag])
            & ((times[lag:] - times[:-lag]) == lag * cadence_ns)
        )
        pairs = int(valid.sum())
        correlation = (
            np.corrcoef(values[:-lag][valid], values[lag:][valid])[0, 1]
            if pairs >= 10
            and np.nanstd(values[:-lag][valid]) > 0
            and np.nanstd(values[lag:][valid]) > 0
            else np.nan
        )
        rows.append((lag, pairs, correlation))
    return rows


temporal_rows, effective_rows = [], []
for series_id, frame in series_data.groupby("series_id", sort=True):
    frame = frame.sort_values(timestamp_field)
    timestamps = frame[timestamp_field].dropna()
    unique_ts = timestamps.drop_duplicates()
    positive = unique_ts.diff().dropna()
    positive = positive[positive > pd.Timedelta(0)]
    cadence = positive.median() if len(positive) else pd.NaT
    span = unique_ts.max() - unique_ts.min() if len(unique_ts) else pd.NaT
    expected = (
        int(span / cadence) + 1
        if pd.notna(cadence) and cadence > pd.Timedelta(0) else np.nan
    )
    temporal_rows.append({
        "series_id": series_id,
        "rows": len(frame),
        "start_ts": unique_ts.min(),
        "end_ts": unique_ts.max(),
        "duplicate_timestamps": int(timestamps.duplicated().sum()),
        "median_cadence_seconds": (
            cadence.total_seconds() if pd.notna(cadence) else np.nan
        ),
        "expected_grid_points": expected,
        "observed_grid_coverage": (
            len(unique_ts) / expected if expected and expected > 0 else np.nan
        ),
        "gaps_over_1_5x_cadence": (
            int(positive.gt(cadence * 1.5).sum())
            if pd.notna(cadence) else np.nan
        ),
        "p99_gap_seconds": (
            positive.quantile(0.99).total_seconds() if len(positive) else np.nan
        ),
    })
    if pd.isna(cadence):
        continue
    for field in series_metrics:
        values = pd.to_numeric(frame[field], errors="coerce")
        acf_rows = exact_lag_acf(
            frame[timestamp_field], values, cadence, max_lag=48
        )
        positive_acf = []
        for _, _, correlation in acf_rows:
            if pd.isna(correlation) or correlation <= 0:
                break
            positive_acf.append(correlation)
        tau = 1 + 2 * sum(positive_acf)
        observed_n = int(values.notna().sum())
        effective_rows.append({
            "series_id": series_id,
            "field": field,
            "raw_observed_n": observed_n,
            "lag1_observed_pairs": acf_rows[0][1] if acf_rows else 0,
            "lag1_pair_coverage": (
                acf_rows[0][1] / max(1, observed_n - 1) if acf_rows else np.nan
            ),
            "lag1_acf": acf_rows[0][2] if acf_rows else np.nan,
            "integrated_autocorrelation_time": tau,
            "effective_sample_size": observed_n / tau if tau > 0 else np.nan,
            "unchanged_observed_step_fraction": values.diff().eq(0).mean(),
            "imputation_used": False,
        })

temporal_quality = pd.DataFrame(temporal_rows)
effective_sample_size = pd.DataFrame(effective_rows)
write_csv(temporal_quality, "temporal_quality.csv")
write_csv(effective_sample_size, "effective_sample_size.csv")
display(temporal_quality)
display(effective_sample_size.head(30))


## 12. Where does variance live?

This is a **descriptive marginal ICC screen**, not a fitted causal mixed-effects
model. Each grouping is evaluated separately to ask whether peer structure may carry
signal. Bootstrap intervals resample whole groups, not rows.


In [ ]:
def icc_from_summaries(summary):
    summary = summary.loc[summary["n"].ge(2)].copy()
    group_count = len(summary)
    total_n = summary["n"].sum()
    if group_count < 2 or total_n <= group_count:
        return np.nan
    grand = np.average(summary["mean"], weights=summary["n"])
    ss_between = (summary["n"] * (summary["mean"] - grand) ** 2).sum()
    ss_within = ((summary["n"] - 1) * summary["var"].fillna(0)).sum()
    ms_between = ss_between / (group_count - 1)
    ms_within = ss_within / (total_n - group_count)
    n0 = (
        total_n - (summary["n"] ** 2).sum() / total_n
    ) / (group_count - 1)
    denominator = ms_between + (n0 - 1) * ms_within
    return (ms_between - ms_within) / denominator if denominator > 0 else np.nan


def one_way_icc(frame, field, group_field):
    working = frame[[group_field, field]].copy()
    working[field] = pd.to_numeric(working[field], errors="coerce")
    working = working.dropna()
    summaries = (
        working.groupby(group_field)[field]
        .agg(n="size", mean="mean", var="var")
        .reset_index(drop=True)
    )
    estimate = icc_from_summaries(summaries)
    boot = []
    if len(summaries) >= 3 and BOOTSTRAP_REPS > 0:
        local_rng = np.random.default_rng(RANDOM_SEED)
        for _ in range(BOOTSTRAP_REPS):
            chosen = local_rng.integers(0, len(summaries), len(summaries))
            boot.append(icc_from_summaries(summaries.iloc[chosen].reset_index(drop=True)))
    return {
        "icc_estimate": estimate,
        "bootstrap_p025": np.nanquantile(boot, 0.025) if boot else np.nan,
        "bootstrap_p975": np.nanquantile(boot, 0.975) if boot else np.nan,
        "groups": len(summaries),
        "rows": int(summaries["n"].sum()) if len(summaries) else 0,
    }


group_levels = (
    ["olt_id", "pon_port", "splitter_l2", entity_field]
    if EDA_SECTOR == "telecom"
    else [entity_field]
)
if EDA_SECTOR == "petrobras_3w":
    variance_frame = analysis_sample.copy()
    variance_frame["source_file_id"] = analysis_sample["source_file_id"]
    group_levels.append("source_file_id")
else:
    variance_frame = operational_sample

variance_rows = []
for field in focus_metrics:
    for group_field in group_levels:
        if group_field not in variance_frame:
            continue
        result = one_way_icc(variance_frame, field, group_field)
        variance_rows.append({
            "field": field,
            "grouping": group_field,
            **result,
            "interpretation": "marginal one-way ICC; group structure screen",
        })
variance_structure = pd.DataFrame(variance_rows)
write_csv(variance_structure, "variance_structure.csv")
display(variance_structure)


In [ ]:
peer_rows = []
if EDA_SECTOR == "telecom":
    peer_lookup = selected_entity_table.set_index("ont_id")
    for field in focus_metrics:
        pivot = series_data.pivot_table(
            index=timestamp_field,
            columns=entity_field,
            values=field,
            aggfunc="first",
        )
        correlations = pivot.corr(method="spearman", min_periods=30)
        entities = list(correlations.columns)
        for left_index, left in enumerate(entities):
            for right in entities[left_index + 1:]:
                lrow, rrow = peer_lookup.loc[left], peer_lookup.loc[right]
                if lrow["splitter_l2"] == rrow["splitter_l2"]:
                    relation = "same_splitter_l2"
                elif lrow["pon_port"] == rrow["pon_port"]:
                    relation = "same_pon_different_l2"
                elif lrow["olt_id"] == rrow["olt_id"]:
                    relation = "same_olt_different_pon"
                else:
                    relation = "different_olt"
                peer_rows.append({
                    "field": field,
                    "peer_relation": relation,
                    "correlation": correlations.loc[left, right],
                })
    peer_correlation = (
        pd.DataFrame(peer_rows)
        .groupby(["field", "peer_relation"], as_index=False)
        .agg(
            median_spearman=("correlation", "median"),
            pair_count=("correlation", "count"),
        )
    )
else:
    peer_correlation = pd.DataFrame([{
        "field": None,
        "peer_relation": "not available",
        "median_spearman": np.nan,
        "pair_count": 0,
        "reason": "3W provides well identity but no native manifold or peer topology",
    }])
write_csv(peer_correlation, "peer_correlation.csv")
display(peer_correlation)


## 13. Drift, structural-change and recurrence screens

Theil–Sen slopes and block-median jumps are robust descriptive screens. Recurrence
uses exact observed timestamp pairs at candidate lags. For 3W these are short-horizon
recurrence candidates, **not calendar seasonality**.


In [ ]:
candidate_lags = (
    {"1_hour": pd.Timedelta(hours=1), "6_hours": pd.Timedelta(hours=6),
     "12_hours": pd.Timedelta(hours=12), "1_day": pd.Timedelta(days=1),
     "7_days": pd.Timedelta(days=7)}
    if EDA_SECTOR == "telecom"
    else {"10_seconds": pd.Timedelta(seconds=10), "1_minute": pd.Timedelta(minutes=1),
          "5_minutes": pd.Timedelta(minutes=5), "30_minutes": pd.Timedelta(minutes=30),
          "1_hour": pd.Timedelta(hours=1)}
)

drift_rows, recurrence_rows = [], []
for series_id, frame in series_data.groupby("series_id", sort=True):
    frame = frame.sort_values(timestamp_field)
    times = pd.to_datetime(frame[timestamp_field], utc=True)
    elapsed_days = (times - times.min()).dt.total_seconds() / 86400
    for field in focus_metrics:
        values = pd.to_numeric(frame[field], errors="coerce")
        valid = values.notna() & elapsed_days.notna()
        x, y = elapsed_days[valid].to_numpy(), values[valid].to_numpy()
        if len(y) >= 20 and np.nanstd(y) > 0:
            positions = np.linspace(0, len(y) - 1, min(500, len(y)), dtype=int)
            slope, intercept, low, high = stats.theilslopes(y[positions], x[positions])
            blocks = pd.qcut(
                np.arange(len(y)), q=min(8, len(y)), labels=False, duplicates="drop"
            )
            block_medians = pd.Series(y).groupby(blocks).median()
            block_mads = pd.Series(y).groupby(blocks).apply(
                lambda part: np.median(np.abs(part - np.median(part)))
            )
            pooled_mad = np.nanmedian(block_mads)
            jumps = block_medians.diff().abs()
            break_count = int(
                jumps.gt(6 * pooled_mad).sum()
                if pooled_mad > 0 else 0
            )
            drift_rows.append({
                "series_id": series_id,
                "field": field,
                "theil_sen_slope_per_day": slope,
                "slope_ci_low": low,
                "slope_ci_high": high,
                "block_median_range": block_medians.max() - block_medians.min(),
                "pooled_block_mad": pooled_mad,
                "large_adjacent_block_jumps": break_count,
                "screen_not_causal_break_test": True,
            })
        indexed = pd.Series(values.to_numpy(), index=times).dropna()
        indexed = indexed[~indexed.index.duplicated(keep="first")].sort_index()
        for lag_name, lag in candidate_lags.items():
            left = indexed.rename("left")
            right = indexed.copy()
            right.index = right.index + lag
            paired = pd.concat([left, right.rename("right")], axis=1).dropna()
            correlation = (
                paired.corr(method="spearman").iloc[0, 1]
                if len(paired) >= 30
                and paired["left"].nunique() > 1
                and paired["right"].nunique() > 1
                else np.nan
            )
            recurrence_rows.append({
                "series_id": series_id,
                "field": field,
                "candidate_lag": lag_name,
                "observed_pairs": len(paired),
                "spearman_at_exact_lag": correlation,
                "status": "estimable" if pd.notna(correlation) else "not_estimable",
                "method": "exact observed timestamp pairs",
                "imputation_used": False,
                "claim": "calendar candidate" if EDA_SECTOR == "telecom" else "short-horizon recurrence only",
            })

drift_screen = pd.DataFrame(drift_rows)
recurrence_screen = pd.DataFrame(recurrence_rows)
write_csv(drift_screen, "drift_and_change_screen.csv")
write_csv(recurrence_screen, "recurrence_screen.csv")
display(drift_screen.head(30))
display(
    recurrence_screen.sort_values(
        "spearman_at_exact_lag", ascending=False
    ).head(30)
)


## 14. Discrete states and cumulative counters


In [ ]:
state_transition_rows, dwell_rows, counter_rows = [], [], []
if EDA_SECTOR == "petrobras_3w":
    state_fields = [
        field for field in series_metrics if field.startswith("ESTADO-")
    ]
    for series_id, frame in series_data.groupby("series_id", sort=True):
        frame = frame.sort_values(timestamp_field)
        times = frame[timestamp_field]
        cadence = times.drop_duplicates().diff().dropna().median()
        for field in state_fields:
            values = pd.to_numeric(frame[field], errors="coerce")
            previous = values.shift()
            time_gap = times.diff()
            valid_transition = (
                values.notna() & previous.notna()
                & time_gap.le(cadence * 1.5)
            )
            transitions = pd.DataFrame({
                "from_state": previous[valid_transition],
                "to_state": values[valid_transition],
            }).value_counts().rename("count").reset_index()
            for row in transitions.itertuples(index=False):
                state_transition_rows.append({
                    "series_id": series_id,
                    "field": field,
                    "from_state": row.from_state,
                    "to_state": row.to_state,
                    "count": row.count,
                })
            break_run = (
                values.ne(previous)
                | values.isna()
                | previous.isna()
                | time_gap.gt(cadence * 1.5)
            )
            run_id = break_run.cumsum()
            for _, run in frame.assign(_value=values, _run=run_id).groupby("_run"):
                if run["_value"].notna().all() and len(run):
                    dwell_rows.append({
                        "series_id": series_id,
                        "field": field,
                        "state": run["_value"].iloc[0],
                        "observations": len(run),
                        "dwell_seconds": len(run) * cadence.total_seconds(),
                    })
else:
    for series_id, frame in series_data.groupby("series_id", sort=True):
        frame = frame.sort_values(timestamp_field)
        for field in ["uptime_s", "reboot_count"]:
            if field not in frame:
                continue
            values = pd.to_numeric(frame[field], errors="coerce")
            increments = values.diff()
            counter_rows.append({
                "series_id": series_id,
                "field": field,
                "negative_increments_or_resets": int(increments.lt(0).sum()),
                "positive_increments": int(increments.gt(0).sum()),
                "unchanged_steps": int(increments.eq(0).sum()),
                "missing_fraction": values.isna().mean(),
            })

state_transitions = pd.DataFrame(state_transition_rows)
dwell_times = pd.DataFrame(dwell_rows)
counter_diagnostics = pd.DataFrame(counter_rows)
write_csv(state_transitions, "state_transitions.csv")
write_csv(dwell_times, "state_dwell_times.csv")
write_csv(counter_diagnostics, "counter_diagnostics.csv")
display(
    state_transitions.head(30) if len(state_transitions)
    else counter_diagnostics
)
display(
    dwell_times.groupby(["field", "state"])["dwell_seconds"]
    .describe().reset_index().head(30)
    if len(dwell_times) else pd.DataFrame()
)


## 15. Dependence after weighting entities and differencing time

Pooled row correlation can be dominated by long entities and common trends. We report:
pooled Spearman, the equal-entity mean of within-entity Spearman correlations, and
correlation of first differences from complete selected histories. No independent-row
p-values are reported.


In [ ]:
corr_fields = [
    field for field in focus_metrics
    if operational_sample[field].notna().sum() >= 50
    and operational_sample[field].nunique(dropna=True) > 1
]
pooled_sample = operational_sample[corr_fields]
if len(pooled_sample) > 50000:
    pooled_sample = pooled_sample.sample(50000, random_state=RANDOM_SEED)
pooled_correlation = pooled_sample.corr(method="spearman")

entity_corrs = []
for _, frame in operational_sample.groupby(entity_field):
    if len(frame) >= 20:
        entity_corrs.append(frame[corr_fields].corr(method="spearman"))
entity_balanced_correlation = (
    sum(entity_corrs) / len(entity_corrs)
    if entity_corrs else pd.DataFrame(index=corr_fields, columns=corr_fields)
)

change_parts = []
for _, frame in series_data.groupby("series_id"):
    change_parts.append(
        frame.sort_values(timestamp_field)[corr_fields].apply(
            pd.to_numeric, errors="coerce"
        ).diff()
    )
change_correlation = pd.concat(change_parts).corr(method="spearman")
write_csv(pooled_correlation, "correlation_pooled_rows.csv", index=True)
write_csv(entity_balanced_correlation, "correlation_entity_balanced.csv", index=True)
write_csv(change_correlation, "correlation_first_differences.csv", index=True)

figure, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for axis, matrix, title in [
    (axes[0], pooled_correlation, "pooled rows"),
    (axes[1], entity_balanced_correlation, "equal-entity mean"),
    (axes[2], change_correlation, "first differences"),
]:
    image = axis.imshow(matrix, vmin=-1, vmax=1, cmap="coolwarm")
    axis.set_xticks(range(len(matrix)))
    axis.set_yticks(range(len(matrix)))
    axis.set_xticklabels(matrix.columns, rotation=90)
    axis.set_yticklabels(matrix.index)
    axis.set_title(title)
figure.colorbar(image, ax=axes.ravel().tolist(), fraction=0.02)
save_figure(figure, "04_conditioned_dependence.png")


## 16. Label-free alert-volume feasibility

This is deliberately **not the final model**. It asks whether a history-only robust
score, under several frozen thresholds and persistence rules, would produce a
plausible number of episodes. Counters are differenced, counts use `log1p`, and
discrete states are excluded from this Gaussian-style score.


In [ ]:
feature_kinds = feature_dictionary.set_index("native_field")["measurement_kind"].to_dict()


def alert_transform(values, field):
    numeric = pd.to_numeric(values, errors="coerce")
    kind = feature_kinds.get(field, "")
    if "cumulative counter" in kind:
        return numeric.diff().where(numeric.diff().ge(0))
    if "interval count" in kind:
        return np.log1p(numeric.clip(lower=0))
    if field == "ber":
        positive = numeric.where(numeric.gt(0))
        historical_floor = positive.shift(1).expanding(min_periods=1).min()
        return np.log10(numeric.where(numeric.gt(0), historical_floor / 2))
    return numeric


alert_rows = []
alert_metrics = [
    field for field in focus_metrics
    if "discrete state" not in feature_kinds.get(field, "")
]
for series_id, frame in series_data.groupby("series_id", sort=True):
    frame = frame.sort_values(timestamp_field)
    for field in alert_metrics:
        transformed = alert_transform(frame[field], field)
        times = frame[timestamp_field]
        positive_gaps = times.diff().dropna()
        cadence = positive_gaps[positive_gaps.gt(pd.Timedelta(0))].median()
        segments = (
            times.diff().gt(cadence * 1.5).cumsum()
            if pd.notna(cadence) else pd.Series(0, index=frame.index)
        )
        window = min(96, max(24, len(frame) // 20))
        minimum = min(window, max(12, window // 3))
        history = transformed.groupby(segments).shift(1)
        center = history.groupby(segments).transform(
            lambda part: part.rolling(window, min_periods=minimum).median()
        )
        q25 = history.groupby(segments).transform(
            lambda part: part.rolling(window, min_periods=minimum).quantile(0.25)
        )
        q75 = history.groupby(segments).transform(
            lambda part: part.rolling(window, min_periods=minimum).quantile(0.75)
        )
        scale = (q75 - q25) / 1.349
        score = (transformed - center).abs() / scale.replace(0, np.nan)
        dates = frame[timestamp_field].dt.floor("D")
        entity_days = max(1, dates.nunique())
        for threshold in [3.0, 4.0, 5.0]:
            raw_flag = score.ge(threshold)
            for persistence in [1, 2, 4]:
                persistent = raw_flag.groupby(segments).transform(
                    lambda part: part.rolling(
                        persistence, min_periods=persistence
                    ).sum().ge(persistence)
                )
                previous = persistent.groupby(segments).shift(1).fillna(False)
                episode_start = persistent & ~previous
                alert_rows.append({
                    "series_id": series_id,
                    "field": field,
                    "threshold": threshold,
                    "persistence_points": persistence,
                    "scored_points": int(score.notna().sum()),
                    "flagged_points": int(persistent.sum()),
                    "episodes": int(episode_start.sum()),
                    "entity_days": entity_days,
                    "episodes_per_entity_day": episode_start.sum() / entity_days,
                    "history_only": True,
                    "truth_used": False,
                })

alert_detail = pd.DataFrame(alert_rows)
alert_feasibility = (
    alert_detail.groupby(
        ["field", "threshold", "persistence_points"], as_index=False
    )
    .agg(
        scored_points=("scored_points", "sum"),
        flagged_points=("flagged_points", "sum"),
        episodes=("episodes", "sum"),
        entity_days=("entity_days", "sum"),
    )
)
alert_feasibility["episodes_per_entity_day"] = (
    alert_feasibility["episodes"] / alert_feasibility["entity_days"]
)
write_csv(alert_detail, "alert_feasibility_by_series.csv")
write_csv(alert_feasibility, "alert_feasibility_summary.csv")
display(
    alert_feasibility.sort_values(
        ["threshold", "persistence_points", "episodes_per_entity_day"],
        ascending=[True, True, False],
    )
)


## 17. What was generic and what required a sector branch?


In [ ]:
capability_matrix = pd.DataFrame([
    {"capability": "sealed cold-entity holdout", "generic_core": True, "telecom": "ONT", "petrobras_3w": "well", "distortion_or_gap": None},
    {"capability": "warm future holdout", "generic_core": False, "telecom": "global panel time split", "petrobras_3w": "not used", "distortion_or_gap": "unrelated event files make a global time split invalid"},
    {"capability": "unseen-instance holdout", "generic_core": False, "telecom": "not needed for panel", "petrobras_3w": "file split within development wells", "distortion_or_gap": None},
    {"capability": "feature roles and measurement kinds", "generic_core": True, "telecom": "pack supplies meanings", "petrobras_3w": "dataset.ini supplies meanings", "distortion_or_gap": None},
    {"capability": "peer topology", "generic_core": True, "telecom": "OLT/PON/splitter plus geo edge", "petrobras_3w": "well identity only", "distortion_or_gap": "no native manifold topology; none invented"},
    {"capability": "seasonality", "generic_core": False, "telecom": "calendar candidates", "petrobras_3w": "short recurrence only", "distortion_or_gap": "3W event files do not support fleet calendar seasonality"},
    {"capability": "discrete state analysis", "generic_core": True, "telecom": "counter resets", "petrobras_3w": "valve transitions and dwell", "distortion_or_gap": None},
    {"capability": "alert-volume screen", "generic_core": True, "telecom": "entity-day", "petrobras_3w": "file-series-day", "distortion_or_gap": "neither source estimates deployment prevalence"},
])
write_csv(capability_matrix, "cross_sector_capability_matrix.csv")
display(capability_matrix)


## 18. Decision register and pre-label freeze

Every statement carries an epistemic status. The manifest below hashes every
pre-label artifact. After it is written, the notebook may read labels for an audit,
but it may not revise the frozen evidence in the same run.


In [ ]:
decision_register = pd.DataFrame([
    {
        "statement": "The row grain is explicit.",
        "epistemic_status": "exact source fact",
        "estimate_or_evidence": grain,
        "uncertainty": None,
        "frame": "full source metadata",
        "owner": "data science",
        "status": "accepted",
        "consequence": "duplicate keys can be interpreted correctly",
    },
    {
        "statement": "The operational sample represents the sealed exploration frame.",
        "epistemic_status": "sample design assessment",
        "estimate_or_evidence": sampling_validity.to_dict(orient="records"),
        "uncertainty": "bounded sample; see checks",
        "frame": primary_partition,
        "owner": "data science",
        "status": "review any non-pass row",
        "consequence": "distribution summaries remain estimates",
    },
    {
        "statement": "Peer structure may explain part of feature variation.",
        "epistemic_status": "descriptive estimate",
        "estimate_or_evidence": "variance_structure.csv and peer_correlation.csv",
        "uncertainty": "group bootstrap; marginal ICC only",
        "frame": primary_partition,
        "owner": "data science",
        "status": "candidate modelling requirement",
        "consequence": "compare entity values with relevant peers, not only global limits",
    },
    {
        "statement": "Candidate recurrence is real seasonality.",
        "epistemic_status": "hypothesis, not established",
        "estimate_or_evidence": "recurrence_screen.csv",
        "uncertainty": "observed-pair descriptive correlations",
        "frame": "selected complete histories",
        "owner": "data science plus domain practitioner",
        "status": "requires validation",
        "consequence": "do not remove cycles automatically",
    },
    {
        "statement": "A flagged numerical tail is an operational anomaly.",
        "epistemic_status": "hypothesis, not established",
        "estimate_or_evidence": "quality_flags_sample.csv",
        "uncertainty": "may be bounds, zeros, control actions, or faults",
        "frame": primary_partition,
        "owner": "domain practitioner",
        "status": "unresolved",
        "consequence": "retain and encode; do not delete as outlier",
    },
    {
        "statement": "Daily alert volume is operationally tolerable.",
        "epistemic_status": "product hypothesis",
        "estimate_or_evidence": "alert_feasibility_summary.csv",
        "uncertainty": "source prevalence is not deployment prevalence",
        "frame": "selected histories only",
        "owner": "operator",
        "status": "unresolved",
        "consequence": "thresholds cannot be finalised without practitioner feedback",
    },
])
write_csv(decision_register, "decision_register.csv")

prelabel_manifest = {
    "eda_version": "native_eda_v2",
    "contract_version_available_but_not_used_for_translation": CORE_VERSION,
    "sector": EDA_SECTOR,
    "source_root": str(SOURCE_ROOT),
    "run_id": EDA_RUN_ID,
    "seed": RANDOM_SEED,
    "row_grain": grain,
    "exact_source_rows": exact_row_count,
    "exact_source_files": exact_file_count,
    "primary_exploration_partition": primary_partition,
    "sample_design": sample_design,
    "sample_rows": len(operational_sample),
    "complete_series": int(series_data["series_id"].nunique()),
    "truth_values_read_before_this_manifest": False,
    "large_source_data_copied": False,
    "runtime": {
        "python": sys.version.split()[0],
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "pyarrow": pyarrow.__version__,
        "scipy": scipy.__version__,
    },
}
write_json(prelabel_manifest, "prelabel_manifest.json")
prelabel_hashes_before = sha256_tree(PRELABEL)
write_json(
    prelabel_hashes_before,
    "artifact_hashes_prelabel.json",
    folder=OUTPUT,
)
display(decision_register)
print(f"Frozen {len(prelabel_hashes_before)} pre-label artifacts.")


## 19. Evaluation-only label audit — after the freeze

This section reopens exactly the sampled positions and adds labels only inside
`label_audit/`. It may reveal that a hypothesis was wrong, but the pre-label analysis
is not rewritten. A changed hypothesis requires a new run/version and untouched
holdout evidence.


In [ ]:
label_rows = []
if INCLUDE_TRUTH_AUDIT:
    LABEL_AUDIT.mkdir(parents=True)
    if EDA_SECTOR == "telecom":
        available_truth = [
            field for field in truth_fields
            if field in panel_parquet.schema_arrow.names
        ]
        truth_parts = []
        for row_group, positions in analysis_sample.groupby("_row_group"):
            frame = panel_parquet.read_row_group(
                int(row_group), columns=available_truth
            ).to_pandas()
            native_positions = positions["_native_position"].astype(int).to_numpy()
            truth_parts.append(frame.iloc[native_positions].copy())
        truth_sample = pd.concat(truth_parts, ignore_index=True)
        audit_fields = [
            field for field in ["gt_state", "gt_fault_type", "gt_fault_id"]
            if field in truth_sample
        ]
        for field in audit_fields:
            counts = truth_sample[field].astype("string").value_counts(dropna=False)
            for label, count in counts.items():
                label_rows.append({
                    "truth_dimension": field,
                    "label": str(label),
                    "sample_rows": int(count),
                    "sample_fraction": count / len(truth_sample),
                })
    else:
        path_by_id = dict(zip(file_inventory["file_id"], file_inventory["path"]))
        truth_parts = []
        for file_id, positions in analysis_sample.groupby("source_file_id"):
            path = path_by_id[file_id]
            frame = read_threew(path, ["timestamp", "class", "state"])
            native_positions = positions["_native_position"].astype(int).to_numpy()
            selected = frame.iloc[native_positions].copy()
            selected["event_directory"] = int(path.parent.name)
            truth_parts.append(selected)
        truth_sample = pd.concat(truth_parts, ignore_index=True)
        for field in ["event_directory", "class", "state"]:
            counts = truth_sample[field].astype("string").value_counts(dropna=False)
            for label, count in counts.items():
                label_rows.append({
                    "truth_dimension": field,
                    "label": str(label),
                    "sample_rows": int(count),
                    "sample_fraction": count / len(truth_sample),
                })
    label_audit_summary = pd.DataFrame(label_rows)
    write_csv(label_audit_summary, "label_audit_summary.csv", folder=LABEL_AUDIT)
    display(label_audit_summary)
else:
    label_audit_summary = pd.DataFrame(
        columns=["truth_dimension", "label", "sample_rows", "sample_fraction"]
    )
    print("Truth audit disabled. No label value or event directory was read.")


In [ ]:
prelabel_hashes_after = sha256_tree(PRELABEL)
assert prelabel_hashes_after == prelabel_hashes_before, (
    "Pre-label evidence changed after labels were opened."
)

final_report = {
    "eda_version": "native_eda_v2",
    "sector": EDA_SECTOR,
    "output_root": str(OUTPUT),
    "exact_rows": exact_row_count,
    "exact_files": exact_file_count,
    "exact_primary_entities": exact_entity_count,
    "sample_rows": len(operational_sample),
    "sample_entities": int(operational_sample[entity_field].nunique()),
    "complete_series": int(series_data["series_id"].nunique()),
    "quality_flags": len(quality_flags),
    "truth_audit": "completed after freeze" if INCLUDE_TRUTH_AUDIT else "not read",
    "prelabel_artifact_count": len(prelabel_hashes_before),
    "prelabel_hashes_unchanged_after_label_audit": True,
    "next_notebook": (
        "01_TELECOM_WEEK1_END_TO_END.ipynb"
        if EDA_SECTOR == "telecom"
        else "02_PETROBRAS_3W_CONTRACT_CHALLENGE.ipynb"
    ),
}
write_json(final_report, "eda_final_report.json", folder=OUTPUT)
display(pd.Series(final_report, name="value").to_frame())
print("EDA complete. Outputs:", OUTPUT)


## 20. How to hand off the findings

Before continuing:

1. Read `feature_dictionary.csv` and confirm every operational feature has a valid
   measurement kind and nuisance-mode list.
2. Review any non-pass row in `sampling_validity.csv`.
3. Use `variance_structure.csv`, `peer_correlation.csv`, and
   `effective_sample_size.csv` to decide which peer and temporal baselines the model
   must support.
4. Treat recurrence as a candidate, not a fact.
5. Show `alert_feasibility_summary.csv` to a practitioner and ask what incident
   volume is actionable.
6. Keep the holdouts closed. Notebook 01 translates telecom; Notebook 02 challenges
   the contract with real 3W wells; Notebook 03 builds the first ranked incident list.

When no domain expert is immediately available, record assumptions rather than
inventing certainty. Use manuals, source documentation, and conservative statistical
checks, then mark the unresolved operational questions for later validation.
